# 03 · Data Augmentation & Class Balancing

1. Augmented variants of one training image (`src/data/augmentation.py`)
2. Class weights for different balancing strengths
3. Class distribution seen by the model: original vs class-weighted vs weighted sampler

Augmentation is applied to the **training set only**. Figures go to `outputs/figures/`.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
if Path("/kaggle/input").exists() and "DR_DATA_ROOT" not in os.environ:
    hits = [p.parent for p in Path("/kaggle/input").rglob("No_DR") if p.is_dir()]
    if hits:
        os.environ["DR_DATA_ROOT"] = str(hits[0].parent)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from src.data.augmentation import apply_augmentation, get_train_transforms
from src.data.dataset import DRDataset, class_weights, make_sampler
from src.utils.config import load_config
from src.utils.seed import set_seed

cfg = load_config(ROOT / "configs" / "config.yaml")
set_seed(cfg["seed"])
P, CLASS_NAMES, K = cfg["paths"], cfg["data"]["class_names"], cfg["data"]["num_classes"]
FIG = P["figures_dir"]; FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
train = pd.read_csv(P["splits_dir"] / "train.csv", dtype={"patient_id": str})
labels = train["label"].to_numpy()
print(len(train), "training images")

## 1. Augmentation examples

In [ ]:
ds = DRDataset(train[train["label"] == 2].sample(1, random_state=cfg["seed"]), cfg)
img = ds.load_preprocessed(0)
aug = get_train_transforms(cfg)

fig, axes = plt.subplots(2, 5, figsize=(13, 6.2))
axes = axes.ravel()
axes[0].imshow(img); axes[0].set_title("Preprocessed input", fontsize=10)
for ax in axes[1:]:
    ax.imshow(apply_augmentation(aug, img.copy()))
for i, ax in enumerate(axes):
    ax.axis("off")
    if i: ax.set_title(f"Augmented #{i}", fontsize=10)
plt.suptitle("Random augmentation: rotation, flips, zoom/shift, brightness/contrast, small hue shift")
plt.tight_layout(h_pad=2.0); plt.savefig(FIG / "augmentation_examples.png", dpi=200, bbox_inches="tight"); plt.show()

## 2. Class weights

w_c = (N / (K · n_c))^power, rescaled so the average weight per image is 1.
- power 1.0: full inverse frequency. The rarest class gets a very large weight, which risks over-fitting a few hundred images.
- power 0.5 (used): square root. Still strongly favours rare classes, but less extreme.

In [ ]:
counts = np.bincount(labels, minlength=K)
wt = pd.DataFrame({"Stage": CLASS_NAMES, "Train images": counts,
                   "Weight (power 0.5)": class_weights(labels, K, 0.5).round(3),
                   "Weight (power 1.0)": class_weights(labels, K, 1.0).round(3)})
P["metrics_dir"].mkdir(parents=True, exist_ok=True)
wt.to_csv(P["metrics_dir"] / "class_weights.csv", index=False)
wt

## 3. What the model sees in one epoch

- **Original**: plain shuffling.
- **Class weights**: same images, but each class's share of the loss is count × weight.
- **Weighted sampler**: rare-class images are drawn more often (with replacement); simulated for one epoch.

In [ ]:
torch.manual_seed(cfg["seed"])
share = lambda x: 100 * x / x.sum()
dist = pd.DataFrame({
    "Original": share(counts.astype(float)),
    "Class weights (loss share, p=0.5)": share(counts * class_weights(labels, K, 0.5)),
    "Sampler (p=0.5)": share(np.bincount(labels[list(make_sampler(labels, K, 0.5))], minlength=K).astype(float)),
    "Sampler (p=1.0)": share(np.bincount(labels[list(make_sampler(labels, K, 1.0))], minlength=K).astype(float)),
}, index=CLASS_NAMES).round(1)
display(dist)

ax = dist.plot(kind="bar", figsize=(10, 4.2), rot=0, width=0.8)
ax.set_ylabel("% of one training epoch")
ax.set_title("Class distribution seen during training, before and after balancing")
plt.tight_layout(); plt.savefig(FIG / "balancing_distribution.png", dpi=200); plt.show()

## Summary

Figures: `augmentation_examples.png`, `balancing_distribution.png`. Table: `outputs/metrics/class_weights.csv`.
Class weights and the weighted sampler are compared as separate training runs (Phase 6, `experiments.csv`).